# KV grupitöö andmete analüüsimine ja mudeldamine

### Installime ja impordime vajalikud Pythoni teegid

In [1]:
!pip install pandas sqlalchemy
!pip install pandas
!pip install psycopg2-binary

In [2]:
import pandas as pd
from sqlalchemy import create_engine, inspect

### Tõmbame PostgreSQL'ist fact ja dim tabelid Pythonisse

In [3]:
# 1. PostgreSQL andmebaasi ühenduse andmed
DB_USER = "postgres"           # PostgreSQL vaikimisi kasutaja on tavaliselt postgres
DB_PASSWORD = "andmed"     
DB_HOST = "localhost"          # Kui andmebaas jookseb sinu arvutis
DB_PORT = "5432"               # PostgreSQL vaikimisi port
DB_NAME = "postgres"

In [4]:
# 2. Loo ühenduse mootor
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

In [5]:
# Sõnastik, kuhu kõik tabelid salvestatakse
tabelid = {}

In [ ]:

# Määra skeemi nimi, kust andmeid soovid
SIHT_SCHEMA = "grupitoo_kv"

# Määra ära tabelid, mida skeemist soovid

vajalikud_tabelid = ['dim_county', 'fact_kv', 'dim_time', 'fact_palk', 'fact_intressid', 'fact_thi']

# Kood, mis tõmbab kõik tabelid korraga

try:
    with engine.connect() as connection:
        for tabeli_nimi in vajalikud_tabelid:
            query = f'SELECT * FROM "{SIHT_SCHEMA}"."{tabeli_nimi}"'
            
            # Tõmbab mällu ainult valitud tabeli
            tabelid[tabeli_nimi] = pd.read_sql_query(query, connection)
            print(f"✓ Vajalik tabel '{tabeli_nimi}' edukalt tõmmatud! Ridu: {len(tabelid[tabeli_nimi])}")
            
    print("\nValitud tabelid on edukalt mälus ja valmis kasutamiseks!")

except Exception as e:
    print(f"Tekkis viga: {e}. Kontrolli, kas tabelite nimed nimekirjas on kirjutatud õigesti!")

✓ Vajalik tabel 'dim_county' edukalt tõmmatud! Ridu: 15
✓ Vajalik tabel 'fact_kv' edukalt tõmmatud! Ridu: 1289
✓ Vajalik tabel 'dim_time' edukalt tõmmatud! Ridu: 86
✓ Vajalik tabel 'fact_palk' edukalt tõmmatud! Ridu: 1260
✓ Vajalik tabel 'fact_intressid' edukalt tõmmatud! Ridu: 86
✓ Vajalik tabel 'fact_thi' edukalt tõmmatud! Ridu: 84

Valitud tabelid on edukalt mälus ja valmis kasutamiseks!


### Eraldame kõik sissetõmmatud tabelid üksteisest

In [7]:
dim_time = tabelid["dim_time"]
dim_county = tabelid["dim_county"]
fact_kv = tabelid["fact_kv"]
fact_palk = tabelid["fact_palk"]
fact_thi = tabelid["fact_thi"]
fact_intressid = tabelid["fact_intressid"]

### Ühendame (joinime) fact ja dim tabelid

In [8]:
# 1. Alustame peamisest kinnisvara faktitabelist ja lisame sellele dimensioonid
df_koond = pd.merge(
    fact_kv,
    dim_county,
    left_on="fk_county_id",
    right_on="pk_county_id",
    how="left",
)
df_koond = pd.merge(
    df_koond, dim_time, left_on="fk_quarter_id", right_on="pk_quarter_id", how="left"
)

In [9]:
# 2. Lisame palga andmed (ühildame nii asukoha kui ka kvartali järgi)
df_koond = pd.merge(
    df_koond,
    fact_palk,
    on=["fk_county_id", "fk_quarter_id"],
    how="left",
    suffixes=("", "_palk"),
)

In [10]:
# 3. Lisame tarbijahinnaindeksi (THI) andmed (ühildame ainult kvartali järgi)
df_koond = pd.merge(
    df_koond, fact_thi, on="fk_quarter_id", how="left", suffixes=("", "_thi")
)

In [11]:
# 4. Lisame intresside andmed (ühildame ainult kvartali järgi)
df_koond = pd.merge(
    df_koond,
    fact_intressid,
    on="fk_quarter_id",
    how="left",
    suffixes=("", "_intress"),
)

In [12]:
# --- TULEMUSE KONTROLL ---
print(f"Koondtabeli kuju (ridu, veerge): {df_koond.shape}")
print("\nEsimesed 5 rida lõplikust analüüsitabelist:")
display(df_koond.head())

Koondtabeli kuju (ridu, veerge): (1289, 14)

Esimesed 5 rida lõplikust analüüsitabelist:


,fk_quarter_id,transaction_count,avg_price_m2,fk_county_id,pk_county_id,county_name,pk_quarter_id,year_number,quarter_number,quarter_string,half_year,average_salary,keskmine_indeks,intress
0,2005-01-01,2946,758.0,1,1,Harju maakond,2005-01-01,2005,1,2005 I,1,552.0,135.86,3.42
1,2005-01-01,8,163.0,2,2,Hiiu maakond,2005-01-01,2005,1,2005 I,1,400.0,135.86,3.42
2,2005-01-01,406,117.0,3,3,Ida-Viru maakond,2005-01-01,2005,1,2005 I,1,356.0,135.86,3.42
3,2005-01-01,52,132.0,5,5,Järva maakond,2005-01-01,2005,1,2005 I,1,390.0,135.86,3.42
4,2005-01-01,59,84.0,4,4,Jõgeva maakond,2005-01-01,2005,1,2005 I,1,394.0,135.86,3.42
